# Notebook 08: Recommendation System

## Objective

This notebook develops a hybrid product recommendation system for the RetailPulse project.

The recommendation engine combines:

- Popularity-Based Recommendations
- Customer Segment-Based Recommendations
- Personalized Customer Recommendations

The generated recommendations help retail businesses improve customer engagement, increase cross-selling opportunities, and enhance customer retention.

---

### Inputs

- retail_cleaned.csv
- customer_segments.csv
- high_risk_customers.csv (optional)

### Outputs

- customer_product_recommendations.csv
- top_products_by_segment.csv
- recommendation_summary.csv

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", None)

In [2]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

REPORT_DIR = PROJECT_ROOT / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project Root :", PROJECT_ROOT)
print("Processed Data :", PROCESSED_DIR)
print("Reports :", REPORT_DIR)

Project Root : c:\N_VsCode\Zidio Project\PROJECT\RetailPulse-AI-Customer-Analytics
Processed Data : c:\N_VsCode\Zidio Project\PROJECT\RetailPulse-AI-Customer-Analytics\data\processed
Reports : c:\N_VsCode\Zidio Project\PROJECT\RetailPulse-AI-Customer-Analytics\reports


In [6]:
# ============================================================
# Load Required Datasets
# ============================================================

# Main cleaned retail dataset
retail_df = pd.read_csv(
    PROCESSED_DIR / "retail_cleaned.csv",
    parse_dates=["InvoiceDate"]
)

# Customer segmentation output
segments_df = pd.read_csv(
    PROCESSED_DIR / "customer_segments.csv"
)

# High-risk customers
churn_df = pd.read_csv(
    REPORT_DIR / "high_risk_customers.csv"
)

print("✓ Retail dataset loaded")
print("✓ Customer segments loaded")
print("✓ High-risk customers loaded")

✓ Retail dataset loaded
✓ Customer segments loaded
✓ High-risk customers loaded


In [7]:
# ============================================================
# Validate Loaded Data
# ============================================================

print("=" * 60)

print("Retail Dataset")
print(retail_df.shape)
display(retail_df.head())

print("=" * 60)

print("Customer Segments")
print(segments_df.shape)
display(segments_df.head())

print("=" * 60)

if churn_df is not None:
    print("High-Risk Customers")
    print(churn_df.shape)
    display(churn_df.head())

print("=" * 60)

print("Retail Missing Values")
display(retail_df.isnull().sum())

print("=" * 60)

print("Customer Segment Missing Values")
display(segments_df.isnull().sum())

Retail Dataset
(779425, 22)


,InvoiceID,StockCode,ProductDescription,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,InvoiceYear,InvoiceQuarter,InvoiceMonth,MonthName,InvoiceWeek,InvoiceDay,DayName,InvoiceHour,IsWeekend,CustomerCountry,InvoiceMonthYear,BasketSize,BasketValue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.3
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.3
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.3
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.3
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.3


Customer Segments
(5878, 3)


,CustomerID,Cluster,Segment
0,12346.0,2,Champions
1,12347.0,3,Loyal Customers
2,12348.0,3,Loyal Customers
3,12349.0,3,Loyal Customers
4,12350.0,1,At Risk Customers


High-Risk Customers
(3414, 16)


,CustomerID,Recency,Frequency,Monetary,TotalQuantity,AverageOrderValue,UniqueProducts,UniqueCountries,FirstPurchase,LastPurchase,CustomerLifetimeDays,ActiveMonths,RevenuePerMonth,AvgQuantityPerOrder,Churn,PredictedChurn
0,12350.0,310,1,334.40,197,19.670588,17,1,2011-02-02 16:01:00,2011-02-02 16:01:00,0,1.0,334.40,197.0,1,1
1,12351.0,375,1,300.93,261,14.330000,21,1,2010-11-29 15:23:00,2010-11-29 15:23:00,0,1.0,300.93,261.0,1,1
2,12353.0,204,2,406.76,212,16.948333,23,1,2010-10-27 12:44:00,2011-05-19 17:47:00,204,6.8,59.82,106.0,1,1
3,12354.0,232,1,1079.40,530,18.610345,58,1,2011-04-21 13:11:00,2011-04-21 13:11:00,0,1.0,1079.40,530.0,1,1
4,12355.0,214,2,947.61,543,27.074571,35,1,2010-05-21 11:59:00,2011-05-09 13:49:00,353,11.8,80.31,271.5,1,1


Retail Missing Values


InvoiceID             0
StockCode             0
ProductDescription    0
Quantity              0
InvoiceDate           0
UnitPrice             0
CustomerID            0
Country               0
TotalAmount           0
InvoiceYear           0
InvoiceQuarter        0
InvoiceMonth          0
MonthName             0
InvoiceWeek           0
InvoiceDay            0
DayName               0
InvoiceHour           0
IsWeekend             0
CustomerCountry       0
InvoiceMonthYear      0
BasketSize            0
BasketValue           0
dtype: int64

Customer Segment Missing Values


CustomerID    0
Cluster       0
Segment       0
dtype: int64

In [9]:
# ============================================================
# Customer–Product Interaction Matrix
# ============================================================

customer_product_matrix = pd.pivot_table(
    retail_df,
    index="CustomerID",
    columns="ProductDescription",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)

print("Customer–Product Matrix Shape:")
print(customer_product_matrix.shape)

display(customer_product_matrix.head())

Customer–Product Matrix Shape:
(5878, 5283)


ProductDescription,DOORMAT UNION JACK GUNS AND ROSES,3 STRIPEY MICE FELTCRAFT,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,ANIMAL STICKERS,BLACK PIRATE TREASURE CHEST,BROWN PIRATE TREASURE CHEST,Bank Charges,CAMPHOR WOOD PORTOBELLO MUSHROOM,CHERRY BLOSSOM DECORATIVE FLASK,DOLLY GIRL BEAKER,FAIRY CAKE CANDLES,FLAMINGO LIGHTS,HOME SWEET HOME BLACKBOARD,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,IVORY PAPER CUP CAKE CASES,LARGE SKULL WINDMILL,NEW BAROQUE BLACK BOXES,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,PAINT YOUR OWN CANVAS SET,PEACE WOODEN BLOCK LETTERS,RED SPOT GIFT BAG LARGE,RED/WHITE DOT MINI CASES,RIDGED GLASS T-LIGHT HOLDER,SET 2 TEA TOWELS I LOVE LONDON,SET Of 6 SOLDIER SKITTLES,SILVER CHERRY LIGHTS,SILVER T-LIGHT SETTING,SPACEBOY BABY GIFT SET,STAR T-LIGHT HOLDER,TOADSTOOL BEDSIDE LIGHT,TRELLIS COAT RACK,VINTAGE DESIGN GIFT TAGS,WHITE BAMBOO RIBS LAMPSHADE,WHITE CHERRY LIGHTS,10 COLOUR SPACEBOY PEN,11 PC CERAMIC TEA SET POLKADOT,12 ASS ZINC CHRISTMAS DECORATIONS,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 HANGING EGGS HAND PAINTED,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 MINI TOADSTOOL PEGS,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE POSY,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE RED SPOTTY,12 PENCILS SMALL TUBE SKULL,12 PENCILS TALL TUBE POSY,12 PENCILS TALL TUBE RED RETROSPOT,12 PENCILS TALL TUBE RED SPOTTY,12 PENCILS TALL TUBE SKULLS,12 PENCILS TALL TUBE WOODLAND,12 PINK HEN+CHICKS IN BASKET,12 PINK ROSE PEG PLACE SETTINGS,12 RED ROSE PEG PLACE SETTINGS,15 PINK FLUFFY CHICKS IN BOX,15CM CHRISTMAS GLASS BALL 20 LIGHTS,16 PC CUTLERY SET PANTRY DESIGN,16 PIECE CUTLERY SET PANTRY DESIGN,18PC WOODEN CUTLERY SET DISPOSABLE,2 DAISIES HAIR COMB,2 PICTURE BOOK EGGS EASTER BUNNY,2 PICTURE BOOK EGGS EASTER CHICKS,2 PICTURE BOOK EGGS EASTER DUCKS,20 DOLLY PEGS RETROSPOT,200 BENDY SKULL STRAWS,200 RED + WHITE BENDY STRAWS,24 HANGING EASTER EGGS FLORAL TUB,3 BIRDS CANVAS SCREEN,3 BLACK CATS W HEARTS BLANK CARD,3 DRAWER ANTIQUE WHITE WOOD CABINET,3 GARDENIA MORRIS BOXED CANDLES,3 HEARTS HANGING DECORATION RUSTIC,3 HOOK HANGER MAGIC GARDEN,3 HOOK PHOTO SHELF ANTIQUE WHITE,3 PIECE JIGSAW TOADSTOOLS,3 PIECE SPACEBOY COOKIE CUTTER SET,3 PINK HEN+CHICKS IN BASKET,3 RAFFIA RIBBONS 50'S CHRISTMAS,3 RAFFIA RIBBONS VINTAGE CHRISTMAS,3 ROSE MORRIS BOXED CANDLES,3 STRIPEY MICE FELTCRAFT,3 TIER CAKE TIN GREEN AND CREAM,3 TIER CAKE TIN RED AND CREAM,3 TIER SWEETHEART GARDEN SHELF,3 TRADITIONAL COOKIE CUTTERS SET,3 TRADITIONAl BISCUIT CUTTERS SET,3 WHITE CHOC MORRIS BOXED CANDLES,3 WICK CHRISTMAS BRIAR CANDLE,36 DOILIES DOLLY GIRL,36 DOILIES SPACEBOY DESIGN,36 DOILIES VINTAGE CHRISTMAS,36 FOIL HEART CAKE CASES,36 FOIL STAR CAKE CASES,36 PENCILS TUBE POSY,36 PENCILS TUBE RED RETROSPOT,36 PENCILS TUBE RED SPOTTY,36 PENCILS TUBE SKULLS,36 PENCILS TUBE WOODLAND,3D CHRISTMAS STAMPS STICKERS,3D DOG PICTURE PLAYING CARDS,3D HEARTS HONEYCOMB PAPER GARLAND,3D SHEET OF CAT STICKERS,3D SHEET OF DOG STICKERS,3D SHEET OF SEA WORLD STICKERS,3D STICKERS CHRISTMAS STAMPS,3D STICKERS TRADITIONAL CHRISTMAS,3D STICKERS VINTAGE CHRISTMAS,3D TRADITIONAL CHRISTMAS STICKERS,3D VINTAGE CHRISTMAS STICKERS,4 BLUE DINNER CANDLES SILVER FLOCK,4 BURGUNDY WINE DINNER CANDLES,4 FESTIVE GREEN DINNER CANDLES,4 GOLD FLOCK CHRISTMAS BALLS,4 IVORY DINNER CANDLES GOLD FLOCK,4 IVORY DINNER CANDLES SILVER FLOCK,4 LAVENDER BOTANICAL DINNER CANDLES,4 LILY BOTANICAL DINNER CANDLES,4 PEAR BOTANICAL DINNER CANDLES,4 PINK DINNER CANDLE SILVER FLOCK,4 PINK FLOCK CHRISTMAS BALLS,4 ROSE PINK DINNER CANDLES,4 SKY BLUE DINNER CANDLES,4 TRADITIONAL SPINNING TOPS,4 VANILLA BOTANICAL CANDLES,4 WILDFLOWER BOTANICAL CANDLES,5 HOOK BLACKBOARD ORGANISER,5 HOOK HANGER MAGIC TOADSTOOL,5 HOOK HANGER RED MAGIC TOADSTOOL,5 STRAND GLASS NECKLACE AMBER,5 STRAND GLASS NECKLACE AMETHYST,5 STRAND GLASS NECKLACE BLACK,5 STRAND GLASS NECKLACE CRYSTAL,50'S CHRISTMAS PAPER GIFT BAG,50CM ME

In [11]:
# ============================================================
# Merge Customer Segments
# ============================================================

recommendation_df = retail_df.merge(
    segments_df,
    on="CustomerID",
    how="left"
)

print("Merged Dataset Shape:", recommendation_df.shape)

recommendation_df[
    ["CustomerID", "Segment", "ProductDescription", "Quantity"]
].head()

Merged Dataset Shape: (779425, 24)


,CustomerID,Segment,ProductDescription,Quantity
0,13085.0,Loyal Customers,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12
1,13085.0,Loyal Customers,PINK CHERRY LIGHTS,12
2,13085.0,Loyal Customers,WHITE CHERRY LIGHTS,12
3,13085.0,Loyal Customers,"RECORD FRAME 7"" SINGLE SIZE",48
4,13085.0,Loyal Customers,STRAWBERRY CERAMIC TRINKET BOX,24


In [12]:
# ============================================================
# Most Popular Products by Customer Segment
# ============================================================

segment_recommendations = (
    recommendation_df
    .groupby(["Segment", "ProductDescription"])["Quantity"]
    .sum()
    .reset_index()
)

segment_recommendations = (
    segment_recommendations
    .sort_values(
        ["Segment", "Quantity"],
        ascending=[True, False]
    )
)

top_products = (
    segment_recommendations
    .groupby("Segment")
    .head(10)
)

top_products.head(20)

,Segment,ProductDescription,Quantity
1454,At Risk Customers,GIRLS ALPHABET IRON ON PATCHES,4032
3669,At Risk Customers,WHITE HANGING HEART T-LIGHT HOLDER,3841
266,At Risk Customers,ASSTD DESIGN BUBBLE GUM RING,3330
1919,At Risk Customers,LUNCH BAG BLACK SKULL.,3232
3760,At Risk Customers,WORLD WAR 2 GLIDERS ASSTD DESIGNS,3219
573,At Risk Customers,BOYS ALPHABET IRON ON PATCHES,3168
2310,At Risk Customers,PERIWINKLE T-LIGHT HOLDER,2988
1920,At Risk Customers,LUNCH BAG CARS BLUE,2941
1928,At Risk Customers,LUNCH BAG WOODLAND,2927
2222,At Risk Customers,PACK OF 72 RETRO SPOT CAKE CASES,2821


In [13]:
# ============================================================
# Recommend Popular Products for High-Risk Customers
# ============================================================

high_risk_ids = churn_df["CustomerID"].unique()

recommendations = []

for customer in high_risk_ids:

    segment = segments_df.loc[
        segments_df["CustomerID"] == customer,
        "Segment"
    ]

    if len(segment) == 0:
        continue

    segment = segment.values[0]

    products = (
        top_products[
            top_products["Segment"] == segment
        ]
        .head(5)["ProductDescription"]
        .tolist()
    )

    recommendations.append({
        "CustomerID": customer,
        "Segment": segment,
        "RecommendedProducts": ", ".join(products)
    })

recommendations_df = pd.DataFrame(recommendations)

print("Recommendations Generated:", len(recommendations_df))

recommendations_df.head()

Recommendations Generated: 3414


,CustomerID,Segment,RecommendedProducts
0,12350.0,At Risk Customers,"GIRLS ALPHABET IRON ON PATCHES , WHITE HANGING..."
1,12351.0,At Risk Customers,"GIRLS ALPHABET IRON ON PATCHES , WHITE HANGING..."
2,12353.0,Potential Customers,"WORLD WAR 2 GLIDERS ASSTD DESIGNS, ASSORTED CO..."
3,12354.0,Potential Customers,"WORLD WAR 2 GLIDERS ASSTD DESIGNS, ASSORTED CO..."
4,12355.0,Potential Customers,"WORLD WAR 2 GLIDERS ASSTD DESIGNS, ASSORTED CO..."


In [14]:
# ============================================================
# Recommendation Summary
# ============================================================

print("=" * 60)
print("Recommendation Summary")
print("=" * 60)

print("Total Customers Recommended:",
      recommendations_df["CustomerID"].nunique())

print("Customer Segments Covered:",
      recommendations_df["Segment"].nunique())

print("\nRecommendations per Segment")

recommendations_df["Segment"].value_counts()

Recommendation Summary
Total Customers Recommended: 3414
Customer Segments Covered: 4

Recommendations per Segment


Segment
At Risk Customers      1606
Potential Customers    1130
Loyal Customers         634
Champions                44
Name: count, dtype: int64

In [15]:
# ============================================================
# Save Recommendation Results
# ============================================================

output_path = REPORT_DIR / "customer_product_recommendations.csv"

recommendations_df.to_csv(
    output_path,
    index=False
)

print("Recommendations saved successfully.")
print(output_path)

Recommendations saved successfully.
c:\N_VsCode\Zidio Project\PROJECT\RetailPulse-AI-Customer-Analytics\reports\customer_product_recommendations.csv


In [16]:
# ============================================================
# Notebook Summary
# ============================================================

print("=" * 60)
print("Recommendation System Completed")
print("=" * 60)

print(f"Customers Processed : {recommendations_df.shape[0]}")
print(f"Customer Segments   : {recommendations_df['Segment'].nunique()}")
print(f"Recommendations Saved: {output_path}")

print("\nOutput File:")
print("customer_product_recommendations.csv")

print("\nNext Notebook:")
print("09_dashboard.ipynb")

Recommendation System Completed
Customers Processed : 3414
Customer Segments   : 4
Recommendations Saved: c:\N_VsCode\Zidio Project\PROJECT\RetailPulse-AI-Customer-Analytics\reports\customer_product_recommendations.csv

Output File:
customer_product_recommendations.csv

Next Notebook:
09_dashboard.ipynb
